In [1]:
import os

# Tránh lỗi crash do xung đột OpenMP/MKL ở một số môi trường
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import base64
from io import BytesIO

import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)


torch: 2.4.0+cpu
torchvision: 0.19.0+cpu


In [2]:
# ===========================================================
# 0. Thiết lập đường dẫn output (features/)
# ===========================================================
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "src":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

OUTPUT_DIR = PROJECT_ROOT / "features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = str(OUTPUT_DIR / "features.npy")
IMAGELIST_PATH = str(OUTPUT_DIR / "image_list.txt")

print("CWD        :", cwd)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Output dir :", OUTPUT_DIR)
print("-", FEATURES_PATH)
print("-", IMAGELIST_PATH)


CWD        : /home/hoang/Image-Retrieval-System/src
PROJECT_ROOT: /home/hoang/Image-Retrieval-System
Output dir : /home/hoang/Image-Retrieval-System/features
- /home/hoang/Image-Retrieval-System/features/features.npy
- /home/hoang/Image-Retrieval-System/features/image_list.txt


In [3]:
# ===========================================================
# 1. Tiền xử lý ảnh (giống ImageNet)
# ===========================================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Dùng folder `Data/` cùng cấp với `src/` (ổn định dù CWD là repo-root hay src/)
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif cwd.name == "src":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_ROOT = str(PROJECT_ROOT / "Data")


In [ ]:
# ===========================================================
# 2. Tải toàn bộ CIFAR-10 (train + test)
# ===========================================================
trainset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=False, transform=transform)
testset  = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=transform)

#Tổng số ảnh 
full_loader = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset([trainset, testset]),
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print("Tổng số ảnh:", len(trainset) + len(testset))


Tổng số ảnh: 60000


In [ ]:
# ===========================================================
# 3. Tải ResNet-18 ĐÃ FINE-TUNE + bỏ FC để lấy đặc trưng 512-D
# ===========================================================
import torch.nn as nn

# Sử dụng biến PROJECT_ROOT đã khai báo ở Cell 2 để trỏ tới thư mục model
MODEL_PATH = PROJECT_ROOT / "model" / "resnet18_finetuned_cifar10.pt"
print("Đường dẫn mô hình:", MODEL_PATH)

# 1. Khởi tạo cấu trúc ResNet-18 trống (KHÔNG load weights từ ImageNet)
base_model = models.resnet18(weights=None)

# 2. Sửa lớp Fully Connected (FC) cuối cùng thành 10 classes cho khớp với lúc Fine-tune
num_ftrs = base_model.fc.in_features
base_model.fc = nn.Linear(num_ftrs, 10)

# 3. Load trọng số (weights) từ file bạn đã huấn luyện
# Dùng map_location='cpu' để đảm bảo không bị lỗi văng bộ nhớ nếu máy đang hết VRAM
base_model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))

# 4. Cắt bỏ lớp FC cuối cùng để tạo thành bộ trích xuất đặc trưng (Feature Extractor) 512 chiều
model = torch.nn.Sequential(*list(base_model.children())[:-1])

# 5. Chuyển mô hình sang chế độ đánh giá (eval) và đưa vào thiết bị
model.eval()
device = torch.device("cpu") # Bạn có thể đổi thành "cuda" nếu muốn dùng GPU để trích xuất nhanh hơn
model.to(device)

print("Device:", device)
print("Đã load thành công mô hình ResNet-18 Fine-tuned!")

In [5]:
# ===========================================================
# 3. Tải ResNet-18 pretrained + bỏ FC để lấy đặc trưng 512-D
# ===========================================================
try:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    base_model = models.resnet18(weights=weights)
except Exception:
    # Fallback cho torchvision cũ
    base_model = models.resnet18(pretrained=True)

model = torch.nn.Sequential(*list(base_model.children())[:-1])
model.eval()

device = torch.device("cpu")
model.to(device)

print("Device:", device)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/hoang/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100.0%


Device: cpu


In [6]:
# ===========================================================
# 4. Denormalize để đưa ảnh về màu sắc thật trước khi encode base64
# ===========================================================
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    tensor = tensor.clone()
    tensor = tensor * std + mean
    return torch.clamp(tensor, 0, 1)


In [7]:
# ===========================================================
# 5. Trích xuất đặc trưng + chuyển ảnh sang base64
# ===========================================================
features = []
image_list = []

print("Bắt đầu xử lý 60.000 ảnh CIFAR-10... (có thể mất vài phút trên CPU)\n")

with torch.no_grad():
    for batch_idx, (images, _) in enumerate(full_loader):
        images = images.to(device)

        # Trích xuất đặc trưng
        feat = model(images)  # (batch, 512, 1, 1)
        feat = feat.cpu().numpy().reshape(feat.shape[0], -1)  # (batch, 512)
        features.append(feat)

        # Convert ảnh sang base64 (JPEG)
        for img_tensor in images.cpu():
            img_tensor = denormalize(img_tensor)
            img_pil = transforms.ToPILImage()(img_tensor)
            buffered = BytesIO()
            img_pil.save(buffered, format="JPEG", quality=90)
            img_str = base64.b64encode(buffered.getvalue()).decode()
            image_list.append(img_str)

        if (batch_idx + 1) % 50 == 0:
            processed = (batch_idx + 1) * 128
            print(f"   Đã xử lý: {processed:>6} / 60000 ảnh")


Bắt đầu xử lý 60.000 ảnh CIFAR-10... (có thể mất vài phút trên CPU)

   Đã xử lý:   6400 / 60000 ảnh
   Đã xử lý:  12800 / 60000 ảnh
   Đã xử lý:  19200 / 60000 ảnh
   Đã xử lý:  25600 / 60000 ảnh
   Đã xử lý:  32000 / 60000 ảnh
   Đã xử lý:  38400 / 60000 ảnh
   Đã xử lý:  44800 / 60000 ảnh
   Đã xử lý:  51200 / 60000 ảnh
   Đã xử lý:  57600 / 60000 ảnh


In [ ]:
# ===========================================================
# 6. Lưu xuống ổ cứng
# ===========================================================
features = np.vstack(features)
print("\nFeatures shape:", features.shape)  # kỳ vọng (60000, 512)

np.save(FEATURES_PATH, features)

with open(IMAGELIST_PATH, "w") as f:
    f.write("\n".join(image_list))

print("\n" + "=" * 60)
print("HOÀN TẤT 100%!")
print("Đã lưu:")
print("   →", FEATURES_PATH)
print("   →", IMAGELIST_PATH)



Features shape: (60000, 512)

HOÀN TẤT 100%!
Đã lưu:
   → /home/hoang/Image-Retrieval-System/features/features.npy
   → /home/hoang/Image-Retrieval-System/features/image_list.txt

Bây giờ bạn có thể chạy: python src/main.py
Mở http://127.0.0.1:5000 để dùng web tìm ảnh giống!
